# 09 · Business recommendations, limitations and future work

**Guidelines covered:** Business recommendations · Limitations · Future work

This notebook brings the evidence together for the decision maker. The code cells only read the result tables saved in [`reports/`](../reports/), so this notebook runs anywhere (no Colab, GPU or raw data needed): clone the repository, open this notebook and run it.

In [1]:
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 60)
S, T = '../reports/statistical_tests', '../reports/tables'

## 1. The evidence in four tables

**(a) Fix priorities per app.** Problems that are statistically significant (p < 0.05), mentioned in at least 30 reviews, ranked by the potential gain in the app's average rating if the problem were fixed (from cell M3).

In [2]:
pri = pd.read_csv(f'{S}/priority_matrix.csv')
top = (pri[(pri.p_value < 0.05) & (pri.stars_effect < 0) & (pri.reviews_mentioning >= 30)]
       .sort_values(['app', 'gain_if_fixed'], ascending=[True, False]).groupby('app').head(3))
top[['app', 'theme', 'prevalence_%', 'stars_effect', 'current_avg', 'gain_if_fixed']].reset_index(drop=True)

,app,theme,prevalence_%,stars_effect,current_avg,gain_if_fixed
0,BHIM,"Registration, OTP & SIM verification",19.45,-0.986,2.675,0.182
1,BHIM,App not opening / crashes / updates,15.45,-0.549,2.675,0.079
2,BHIM,Slow performance,11.38,-0.420,2.675,0.044
3,Google Pay,App not opening / crashes / updates,4.45,-1.462,3.989,0.057
4,Google Pay,Cashback & rewards,3.12,-0.963,3.989,0.026
5,Google Pay,Failed / pending payments,2.54,-0.847,3.989,0.020
6,Paytm,App not opening / crashes / updates,2.50,-1.381,4.452,0.028
7,Paytm,Customer support,1.68,-0.972,4.452,0.015
8,Paytm,Security / device-integrity blocks,1.43,-1.106,4.452,0.014
9,PhonePe,App not opening / crashes / updates,1.28,-1.581,4.385,0.017


**(b) Stars lost when a review mentions a problem, all apps (cell M2).** Ordinary least squares with robust standard errors, controlling for the app and review length.

In [3]:
drv = pd.read_csv(f'{S}/rating_drivers_all_apps.csv')
drv.assign(significant=drv.p_value < 0.05)[['theme', 'stars_effect', 'ci_low', 'ci_high', 'p_value', 'significant']]

,theme,stars_effect,ci_low,ci_high,p_value,significant
0,AutoPay / mandates,-1.220,-1.402,-1.037,0.0000,True
1,App not opening / crashes / updates,-1.101,-1.160,-1.042,0.0000,True
2,Security / device-integrity blocks,-1.098,-1.235,-0.961,0.0000,True
3,Fees & charges,-0.860,-1.025,-0.695,0.0000,True
4,"Registration, OTP & SIM verification",-0.844,-0.917,-0.770,0.0000,True
5,Failed / pending payments,-0.760,-0.842,-0.677,0.0000,True
6,Login & account blocked,-0.738,-0.835,-0.641,0.0000,True
7,Customer support,-0.712,-0.795,-0.629,0.0000,True
8,"Ads, notifications & add-ons",-0.681,-0.794,-0.568,0.0000,True
9,Cashback & rewards,-0.661,-0.739,-0.582,0.0000,True


**(c) Releases that were significantly worse than the rest of their app (cell V2)**, and **(d) complaint themes that changed significantly in the last four weeks (cell D5).**

In [4]:
mon = pd.read_csv(f'{S}/release_monitor.csv')
print(mon[mon.significant & (mon['diff'] < 0)][['app', 'version', 'first_seen', 'reviews', 'avg_rating', 'rest_of_app', 'p_value']].to_string(index=False))
em = pd.read_csv(f'{S}/emerging_topics.csv')
print()
print(em[em.significant][['app', 'theme', 'earlier_%', 'last_4_weeks_%', 'change_pp']].to_string(index=False))

       app                             version first_seen  reviews  avg_rating  rest_of_app      p_value
      BHIM                              4.0.26 2026-07-19     2452        2.62         2.77 9.709719e-04
      BHIM                              4.0.24 2026-06-22     1707        2.52         2.78 3.167409e-08
Google Pay 331.1.2 (arm64-v8a_release_flutter) 2026-05-18     1097        3.60         4.02 6.162082e-15
Google Pay 351.1.1 (arm64-v8a_release_flutter) 2026-09-29      444        3.73         4.00 8.613793e-04
     Paytm                             10.84.5 2026-09-02     2486        4.11         4.54 1.378325e-59

    app                                theme  earlier_%  last_4_weeks_%  change_pp
PhonePe  App not opening / crashes / updates        8.4            16.7        8.3
   BHIM               QR scanning & location        2.1             8.7        6.6
   BHIM                   Cashback & rewards        4.4            10.6        6.2
   BHIM                     Slow perf

In [5]:
pd.read_csv(f'{T}/reply_analysis.csv')

,app,reply_rate_all_%,reply_rate_1_2_star_%,reply_rate_5_star_%,median_hours_1_2_star,template_share_1_2_star_%,template_share_5_star_%
0,BHIM,96.8,98.4,93.8,45.7,95.9,96.1
1,Google Pay,11.4,44.1,0.3,0.3,6.5,2.5
2,Paytm,96.2,94.8,96.5,0.4,8.5,86.9
3,PhonePe,25.8,74.6,16.9,1.5,0.0,98.5


## 2. Business recommendations

Each recommendation follows the same logic: **finding → action → evidence → expected benefit**.

### BHIM: fix onboarding first
- **Finding:** registration, OTP and SIM verification is 37–44% of BHIM's specific complaints and appears in 19% of all BHIM reviews; each mention costs about one star (−0.99).
- **Actions:** reliable OTP auto-read with manual entry as a fallback; clear, in-app guidance on the SIM and mobile-data rules; individual replies within hours instead of 46-hour templates.
- **Expected benefit:** about **+0.18 stars** on a 2.68-star app, the largest gain of any single fix in the study. Onboarding complaints already fell from 45–50% to 17–20% of BHIM complaints between July and October, so part of the problem appears to be fixed; the remaining gain should be verified with a staged rollout.

### Paytm: safer releases and a recovery path for blocked users
- **Finding:** version 10.84.5 nearly doubled 1-star reviews (9.2% → 17.6%) and raised crash mentions from 1.9% to 9.4%; the next versions brought both back below the earlier level (p < 0.001). Security and device-integrity blocks are Paytm's largest complaint theme (29–36%).
- **Actions:** staged rollouts (release to a small share of users first) combined with a review-based release monitor like the one built here; a clear self-service recovery path for genuine users blocked by security checks.
- **Expected benefit:** each problem affects only 1–3% of reviewers, so the average rating moves little, but every affected user takes about 1 to 1.4 stars off. Catching the next bad release within days protects the rating and the users most likely to switch.

### Google Pay: payment trust
- **Finding:** failed or pending payments are its top theme (24–29%); AutoPay problems cost the most stars of any theme (−1.22); Google Pay answers only 44% of complaints.
- **Actions:** automatic refunds and a status tracker for failed payments; one-tap AutoPay cancellation; answer more complaints, starting with the most up-voted ones.

### PhonePe: reachable support
- **Finding:** customer support is its top theme (about 24%); fees cost PhonePe users more stars than average (−1.51 in the PhonePe model); crash complaints are rising significantly (+8.3 percentage points in the last four weeks).
- **Actions:** a human escalation path for blocked accounts and stuck money; transparent fee communication; investigate the crash rise after version 26.09.25.0 now.

### For every app: a weekly operating rhythm
1. Scrape new reviews weekly and refresh the dashboard.
2. Check the release monitor before widening any rollout.
3. Review the emerging-topics view; investigate any significant rise.
4. Track reply speed and template share, not only reply rate.

## 3. Limitations
- **Self-selection:** unhappy users write more reviews, so the results describe complaints, not the experience of all users.
- **Period and language:** two to five months per app; English and Hinglish only. Reviews in Indian scripts were kept for rating statistics but not for text models.
- **Labels from stars:** star ratings are a noisy label (1,667 hidden complaints); the model also misreads polite, request-style complaints.
- **Association, not causation:** the regressions show how strongly problems are associated with lower ratings; the rating gains are potential, not guaranteed.
- **Unspecified complaints:** 15–43% of complaints, depending on the app, name no specific problem and cannot be assigned to a theme.
- **Keyword themes:** the business models use keyword rules derived from the topic model; a stricter rule changes the crash effect from −1.10 to −0.91 stars (still significant).
- **No internal data:** without crash logs or support tickets, causes of release problems cannot be confirmed.

## 4. Future work (how the project can be improved)
1. **Automation:** weekly scraping and model refresh with alerts when a theme or release crosses a significance threshold.
2. **Better models:** multilingual or Hinglish-specific transformers for code-mixed and Indian-script reviews; add hand-labelled polite complaints to fine-tuning; compare with large language models.
3. **Aspect-based sentiment:** sentiment per theme within a review, instead of one label per review.
4. **Causal evidence:** A/B tests or staged rollouts to measure the real rating effect of a fix.
5. **More sources:** the Apple App Store, social media and more fintech apps; link themes to internal support tickets and payment-failure logs.
6. **Business value:** connect rating changes to downloads, retention and transaction volume to estimate the financial return of each fix.